In [2]:
import os
import zipfile

DRIVE_FOLDER = '/content/drive/MyDrive/Colab Notebooks/mobile-leafdoc'
ZIP_PATH_pd = os.path.join(DRIVE_FOLDER, 'PlantDoc-Dataset.zip')
ZIP_PATH_pv = os.path.join(DRIVE_FOLDER, 'PlantVillage-Dataset.zip')

if not os.path.exists('/content/data'):
    print("Unzipping dataset...")
    with zipfile.ZipFile(ZIP_PATH_pd, 'r') as zip_ref:
        zip_ref.extractall('/content/data')
    with zipfile.ZipFile(ZIP_PATH_pv, 'r') as zip_ref:
        zip_ref.extractall('/content/data')
    print("✅ Unzip complete!")
else:
    print("Data already unzipped.")

if os.path.exists('/content/data/PlantVillage-Dataset/raw/color'):
    print("✅ PV - Found Training Folder!")
    PV_DIR = '/content/data/PlantVillage-Dataset/raw/color'
else:
    print("⚠️ Check your zip structure. Could not find 'PlantVillage-Dataset/raw/color'")

if os.path.exists('/content/data/PlantDoc-Dataset/train'):
    print("✅ PD - Found Training Folder!")
    PD_DIR = '/content/data/PlantDoc-Dataset/train'
else:
    print("⚠️ Check your zip structure. Could not find 'PlantDoc-Dataset/train'")

Unzipping dataset...
✅ Unzip complete!
✅ PV - Found Training Folder!
✅ PD - Found Training Folder!


In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os
import json
import numpy as np
import random
from PIL import Image
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import torch
from collections import Counter

# --- 1. Fourier Domain Adaptation (FDA) ---
def fda_transform(src_img, trg_img, beta=0.01):
    src = np.array(src_img).astype(np.float32)
    trg = np.array(trg_img).astype(np.float32)

    if len(src.shape) == 2: src = np.stack([src]*3, axis=-1)
    if len(trg.shape) == 2: trg = np.stack([trg]*3, axis=-1)

    fft_src = np.fft.fft2(src, axes=(0, 1))
    fft_trg = np.fft.fft2(trg, axes=(0, 1))

    fft_src_shift = np.fft.fftshift(fft_src, axes=(0, 1))
    fft_trg_shift = np.fft.fftshift(fft_trg, axes=(0, 1))

    h, w = fft_src.shape[:2]
    b = int(np.floor(np.amin((h, w)) * beta))
    c_h, c_w = h // 2, w // 2

    fft_src_shift[c_h-b:c_h+b, c_w-b:c_w+b] = fft_trg_shift[c_h-b:c_h+b, c_w-b:c_w+b]

    fft_src_ishift = np.fft.ifftshift(fft_src_shift, axes=(0, 1))
    result = np.fft.ifft2(fft_src_ishift, axes=(0, 1))
    result = np.abs(result).clip(0, 255).astype(np.uint8)

    return Image.fromarray(result)

# --- 2. Custom Noise Transform ---
class AddGaussianNoise(object):
    def __init__(self, mean=0., std=1.):
        self.std = std
        self.mean = mean

    def __call__(self, img):
        if random.random() > 0.5:
            return img
        np_img = np.array(img)
        noise = np.random.normal(self.mean, self.std, np_img.shape)
        noisy_img = np_img + noise
        return Image.fromarray(np.clip(noisy_img, 0, 255).astype('uint8'))

# --- 3. Define Asymmetric Transforms ---
def get_domain_transforms(img_size=224):
    stats = {'mean':[0.485,0.456,0.406], 'std':[0.229,0.224,0.225]}

    pv_transform = transforms.Compose([
        transforms.Resize((256, 256)),
        transforms.RandomCrop((img_size, img_size)),
        transforms.RandomHorizontalFlip(),
        transforms.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.4, hue=0.1),
        transforms.RandomGrayscale(p=0.1),
        transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 2.0)),
        AddGaussianNoise(std=10),
        transforms.ToTensor(),
        transforms.Normalize(**stats)
    ])

    pd_transform = transforms.Compose([
        transforms.Resize((256, 256)),
        transforms.RandomCrop((img_size, img_size)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(15),
        transforms.ColorJitter(brightness=0.2, contrast=0.2),
        transforms.ToTensor(),
        transforms.Normalize(**stats)
    ])

    val_transform = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor(),
        transforms.Normalize(**stats)
    ])

    return pv_transform, pd_transform, val_transform

# --- 4. Dataset Class ---
class RobustMixedDataset(Dataset):
    def __init__(self, pv_dir, pd_dir, classmap_path,
                 pv_transform=None, pd_transform=None,
                 enable_fda=True, fda_beta=0.005):

        self.pv_transform = pv_transform
        self.pd_transform = pd_transform
        self.enable_fda = enable_fda
        self.fda_beta = fda_beta
        self.samples = []
        self.pd_paths = []

        with open(classmap_path, 'r') as f:
            cm = json.load(f)
        pv_map = cm["plantvillage_to_unified"]
        pd_map = cm["plantdoc_to_unified"]

        pv_classes = set(pv_map.values())
        pd_classes = set(pd_map.values())
        common_classes = sorted(list(pv_classes.intersection(pd_classes)))

        self.class_to_idx = {name: i for i, name in enumerate(common_classes)}
        self.unified_names = common_classes

        print(f"✅ Found {len(common_classes)} overlapping classes.")
        self._collect(pv_dir, pv_map, domain_code=0)
        self._collect(pd_dir, pd_map, domain_code=1)

    def _collect(self, root, mapping, domain_code):
        if not os.path.exists(root): return
        count = 0
        for folder in os.listdir(root):
            path = os.path.join(root, folder)
            if not os.path.isdir(path): continue
            label_str = mapping.get(folder)
            if label_str in self.class_to_idx:
                label_idx = self.class_to_idx[label_str]
                for f in os.listdir(path):
                    if f.lower().endswith(('.jpg', '.png', '.jpeg')):
                        full_path = os.path.join(path, f)
                        self.samples.append((full_path, label_idx, domain_code))
                        if domain_code == 1: self.pd_paths.append(full_path)
                        count += 1
        print(f"Loaded {count} images for Domain {domain_code}")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label, domain = self.samples[idx]
        img = Image.open(path).convert('RGB')

        if domain == 0: # PV
            if self.enable_fda and len(self.pd_paths) > 0 and random.random() < 0.5:
                rand_pd = random.choice(self.pd_paths)
                try:
                    pd_ref = Image.open(rand_pd).convert('RGB').resize(img.size)
                    img = fda_transform(img, pd_ref, beta=self.fda_beta)
                except: pass
            if self.pv_transform: img = self.pv_transform(img)
        else: # PD
            if self.pd_transform: img = self.pd_transform(img)

        return img, label

# --- 5. Loader Helper ---
def make_domain_balanced_loader(dataset, batch_size, pd_weight_factor=5.0, num_workers=2):
    targets = [s[1] for s in dataset.samples]
    domains = [s[2] for s in dataset.samples]
    class_counts = Counter(targets)
    weight_per_class = {c: 1.0/count for c, count in class_counts.items()}

    sample_weights = []
    for t, d in zip(targets, domains):
        w = weight_per_class[t]
        if d == 1: w *= pd_weight_factor
        sample_weights.append(w)

    sampler = WeightedRandomSampler(torch.DoubleTensor(sample_weights), len(sample_weights))
    return DataLoader(dataset, batch_size=batch_size, sampler=sampler, num_workers=num_workers, pin_memory=True)

In [11]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import models, transforms, datasets
from PIL import Image
import numpy as np
import random
import json
import os
from tqdm import tqdm

# Use the LOCAL paths we created in the step above
# PD_DIR, PV_DIR, TEST_DIR are defined in the previous cell
CLASSMAP_PATH = '/content/drive/MyDrive/Colab Notebooks/mobile-leafdoc/classmap.json'
OUT_DIR = '/content/drive/MyDrive/Colab Notebooks/mobile-leafdoc/runs/mixed_imagenet_large_2'

# CLASSMAP_PATH = '/content/drive/MyDrive/mobile-leafdoc/classmap.json'
# OUT_DIR = '/content/drive/MyDrive/mobile-leafdoc/runs/mixed_imagenet_large_2'

TEST_DIR = "/content/data/PlantDoc-Dataset/test/"
# OUT_DIR = "/content/drive/MyDrive/Colab Notebooks/mobile-leafdoc/runs/mixed_new/run1"
PD_DIR = '/content/data/PlantDoc-Dataset/train'
PV_DIR = '/content/data/PlantVillage-Dataset/raw/color'

os.makedirs(OUT_DIR, exist_ok=True)

IMG_SIZE = 224
BATCH_SIZE = 64 # T4 GPU can handle 64 or 128 easily
EPOCHS = 10
LR = 1e-4

# Check for GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if device.type == 'cpu':
    print("⚠️ WARNING: using CPU.")

# Get Transforms
tfm_pv, tfm_pd, tfm_val = get_domain_transforms(IMG_SIZE)

# Build Datasets
print("Building Datasets...")
ds_train = RobustMixedDataset(
    PV_DIR, PD_DIR, CLASSMAP_PATH,
    pv_transform=tfm_pv, pd_transform=tfm_pd,
    enable_fda=True, fda_beta=0.005
)
# ds_val = datasets.ImageFolder(TEST_DIR, transform=tfm_val)
ds_val = RobustMixedDataset(
    pv_dir="",                # Empty path (we don't want PV in validation)
    pd_dir=TEST_DIR,          # Point to your PlantDoc Test folder
    classmap_path=CLASSMAP_PATH,

    # Use the VALIDATION transform (No noise, no flip)
    pd_transform=tfm_val,

    # Disable Training Tricks
    enable_fda=False,
    pv_transform=None
)

# Create Loaders
# On Colab (Linux), num_workers=2 or 4 is fine and faster
train_loader = make_domain_balanced_loader(ds_train, BATCH_SIZE, pd_weight_factor=5.0, num_workers=2)
val_loader = DataLoader(ds_val, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f"✅ Validation Set Fixed: {len(ds_val)} images found.")

# Model
model = models.mobilenet_v3_large(weights='IMAGENET1K_V1')
model.classifier[3] = nn.Linear(model.classifier[3].in_features, len(ds_train.class_to_idx))
model = model.to(device)

# Loss & Optimizer
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-2)

Building Datasets...
✅ Found 28 overlapping classes.
Loaded 38542 images for Domain 0
Loaded 2336 images for Domain 1
✅ Found 28 overlapping classes.
Loaded 236 images for Domain 1
✅ Validation Set Fixed: 236 images found.


In [14]:
OLD_MODEL_DIR = "/content/drive/MyDrive/Colab Notebooks/mobile-leafdoc/runs/mixed_imagenet_large"
RESUME_CHECKPOINT = f"{OLD_MODEL_DIR}/mixed_final.pt"

if os.path.exists(RESUME_CHECKPOINT):
    checkpoint = torch.load(RESUME_CHECKPOINT, map_location=device)

    # Load Model Weights
    model.load_state_dict(checkpoint['model'])
    # Load Optimizer State (Crucial for smooth resuming!)
    if 'optimizer' in checkpoint:
        opt.load_state_dict(checkpoint['optimizer'])
        print("✅ Optimizer state loaded (Learning rate/Momentum preserved).")

    # Determine Starting Epoch
    # start_epoch = checkpoint.get('epoch', 0) + 1
    start_epoch=6
    print(f"✅ Resuming from Epoch {start_epoch}")
else:
    print(f"⚠️ Checkpoint not found at {RESUME_CHECKPOINT}. Starting from scratch.")
    start_epoch = 1

✅ Resuming from Epoch 6


In [15]:

# --- 3. Fast Training Loop ---
best_acc = 0.0
print(f"🚀 Training on {device}...")

for epoch in range(start_epoch, EPOCHS + 1):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    # TQDM for progress bar
    pbar = tqdm(train_loader, desc=f"Epoch {epoch}", leave=False)

    for inputs, labels in pbar:
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
        pbar.set_postfix({'loss': f'{loss.item():.4f}'})

    train_acc = 100 * correct / total
    train_loss = running_loss / len(train_loader)

    # Validation
    model.eval()
    val_correct = 0
    val_total = 0
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, predicted = torch.max(outputs.data, 1)
            val_total += labels.size(0)
            val_correct += (predicted == labels).sum().item()


    val_acc = 100 * val_correct / val_total

    print(f"Epoch {epoch}: Loss={train_loss:.4f} | Train Acc={train_acc:.2f}% | Val Acc={val_acc:.2f}%")

    if val_acc > best_acc:
        best_acc = val_acc
        print(f"  --> ⭐ New Best: {best_acc:.2f}%")
        torch.save({'model': model.state_dict(), 'class_to_idx': ds_train.class_to_idx}, f"{OUT_DIR}/best_model.pt")
    torch.save({'model': model.state_dict(), 'class_to_idx': ds_train.class_to_idx}, f"{OUT_DIR}/final_model.pt")

🚀 Training on cuda...


Epoch 6: Loss=0.7210 | Train Acc=98.45% | Val Acc=59.75%
  --> ⭐ New Best: 59.75%


Epoch 7: Loss=0.7112 | Train Acc=98.52% | Val Acc=58.05%


Epoch 8: Loss=0.7038 | Train Acc=98.72% | Val Acc=58.05%


Epoch 9: Loss=0.6977 | Train Acc=98.77% | Val Acc=57.20%


Epoch 10: Loss=0.6896 | Train Acc=99.01% | Val Acc=61.44%
  --> ⭐ New Best: 61.44%
